In [ ]:
#đọc dữ liệu
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "dataset").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("Gốc repo:", ROOT)

tiki = pd.read_csv(ROOT / "dataset" / "tiki_products.csv", dtype={"sku": str})
hasaki = pd.read_csv(ROOT / "dataset" / "hasaki_products.csv", dtype={"sku": str})

raw = pd.concat([tiki, hasaki], ignore_index=True)
print(tiki.shape, hasaki.shape, raw.shape)

Gốc repo: d:\Bigdata_project\ABDA433708_15_HeThongPhanTichTMDT
(3074, 27) (3871, 22) (6945, 31)


In [ ]:
#số liệu trước làm sạch
def stats(df):
    return {
        "Số dòng": len(df),
        "Số cột": df.shape[1],
        "Tổng ô thiếu": int(df.isna().sum().sum()),
        "Dòng trùng (source+product_id)": int(df.duplicated(subset=["source", "product_id"]).sum()),
        "Giá <= 0 hoặc thiếu": int((df["price"].fillna(0) <= 0).sum()),
    }

before = stats(raw)
before

{'Số dòng': 6945,
 'Số cột': 31,
 'Tổng ô thiếu': 54563,
 'Dòng trùng (source+product_id)': 0,
 'Giá <= 0 hoặc thiếu': 0}

In [7]:
#hàm gom nhóm ngành
def _norm(s):
    s = unicodedata.normalize("NFD", str(s).lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.replace("đ", "d").replace("-", " ")

RULES = [
    ("Điện thoại & Máy tính bảng", ["dien thoai", "may tinh bang", "smartphone"]),
    ("Laptop & Thiết bị điện tử",  ["laptop", "tai nghe", "dien tu", "camera", "may anh"]),
    ("Chăm sóc da mặt",  ["cham soc da", "sua rua mat", "serum", "kem duong", "kem chong nang", "mat na", "toner"]),
    ("Chăm sóc tóc & cơ thể", ["cham soc toc", "dau goi", "dau xa", "sua tam", "cham soc co the"]),
    ("Trang điểm",       ["trang diem", "kem nen", "mascara", "son moi", "phan phu"]),
    ("Nước hoa",         ["nuoc hoa"]),
    ("Thực phẩm chức năng", ["thuc pham chuc nang", "vitamin"]),
]

def _match(text):
    for group, kws in RULES:
        if any(k in text for k in kws):
            return group
    return None

def group_category(row):
    # Ưu tiên danh mục; chỉ dùng tên sản phẩm khi danh mục không khớp
    cat = _norm(f"{row.get('category_name', '')} {row.get('category_path', '')}")
    return _match(cat) or _match(_norm(row.get("name", ""))) or "Khác"

In [8]:
#Hàm làm sạch dữ liệu
# Cột bỏ: sku là mã kho; các cột còn lại chỉ có 1 giá trị hoặc rỗng 100% (theo detect_errors)
DROP_COLS = ["sku", "seller_name", "inventory_status", "order_count",
             "favourite_count", "availability", "has_installment"]

def clean(df):
    df = df.copy()
    log = {}

    # 1. Xóa trùng (source + product_id)
    n = len(df)
    df = df.drop_duplicates(subset=["source", "product_id"], keep="first")
    log["Xóa dòng trùng"] = n - len(df)

    # 2. Chuẩn hóa chữ: bỏ khoảng trắng thừa, thống nhất brand, brand thiếu -> "Không rõ"
    df["name"] = df["name"].str.strip()
    key = df["brand"].str.strip().str.lower()
    canon = df.groupby(key)["brand"].agg(lambda s: s.str.strip().mode().iat[0])
    df["brand"] = key.map(canon)
    log["Brand thiếu điền 'Không rõ'"] = int(df["brand"].isna().sum())
    df["brand"] = df["brand"].fillna("Không rõ")

    # 3. Loại dòng giá bán không hợp lệ (.copy() để tránh SettingWithCopyWarning)
    n = len(df)
    df = df[df["price"].fillna(0) > 0].copy()
    log["Loại dòng giá bán <= 0 hoặc thiếu"] = n - len(df)

    # 4. Giá gốc không hợp lệ -> bằng giá bán (không giảm giá)
    bad = df["original_price"].isna() | (df["original_price"] <= 0) | (df["original_price"] < df["price"])
    log["Sửa giá gốc không hợp lệ"] = int(bad.sum())
    df.loc[bad, "original_price"] = df.loc[bad, "price"]

    # 5. Tính lại % giảm giá
    df["discount_rate"] = ((df["original_price"] - df["price"]) / df["original_price"] * 100).round(0)

    # 6. Đánh giá: rating = 0 nghĩa là "chưa có" (kể cả dòng rating 0 nhưng có review)
    df["review_count"] = df["review_count"].fillna(0)
    df.loc[df["rating_average"] == 0, "rating_average"] = np.nan
    df.loc[df["review_count"] == 0, "rating_average"] = np.nan
    df["has_rating"] = df["rating_average"].notna()

    # 7. quantity_sold thiếu -> 0 (giả định, ghi trong báo cáo)
    log["quantity_sold thiếu điền 0"] = int(df["quantity_sold"].isna().sum())
    df["quantity_sold"] = df["quantity_sold"].fillna(0)

    # 8. Cờ ngoại lai giá theo từng sàn (chỉ gắn cờ, không xóa)
    q1 = df.groupby("source")["price"].transform(lambda s: s.quantile(0.25))
    q3 = df.groupby("source")["price"].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    df["is_price_outlier"] = (df["price"] > q3 + 1.5 * iqr) | (df["price"] < q1 - 1.5 * iqr)
    log["Gắn cờ giá ngoại lai"] = int(df["is_price_outlier"].sum())

    # 9. Nhóm ngành chung hai sàn
    df["category_group"] = df.apply(group_category, axis=1)

    # 10. Bỏ cột không dùng
    df = df.drop(columns=DROP_COLS, errors="ignore")
    return df, log

In [9]:
#Chạy và kiểm tra
clean_df, log = clean(raw)
after = stats(clean_df)

print(log)
print(pd.DataFrame({"Trước": before, "Sau": after}))
print(clean_df.groupby("source")["category_group"].value_counts())

{'Xóa dòng trùng': 0, "Brand thiếu điền 'Không rõ'": 496, 'Loại dòng giá bán <= 0 hoặc thiếu': 0, 'Sửa giá gốc không hợp lệ': 154, 'quantity_sold thiếu điền 0': 510, 'Gắn cờ giá ngoại lai': 539}
                                Trước    Sau
Số dòng                          6945   6945
Số cột                             31     27
Tổng ô thiếu                    54563  27045
Dòng trùng (source+product_id)      0      0
Giá <= 0 hoặc thiếu                 0      0
source  category_group            
hasaki  Chăm sóc da mặt               3069
        Chăm sóc tóc & cơ thể          802
tiki    Khác                          2426
        Chăm sóc tóc & cơ thể          503
        Điện thoại & Máy tính bảng      75
        Laptop & Thiết bị điện tử       36
        Thực phẩm chức năng             34
Name: count, dtype: int64


In [10]:
clean_df.shape


(6945, 27)

In [11]:
#lưu file dữ liệu đã làm sạch
OUT = ROOT / "dataset" / "products_clean.csv"
clean_df.to_csv(OUT, index=False, encoding="utf-8-sig")
print("Đã lưu:", OUT, clean_df.shape)

compare = pd.DataFrame({"Trước": before, "Sau": after})
compare.to_csv(ROOT / "dataset" / "cleaning_before_after.csv", encoding="utf-8-sig")
pd.Series(log, name="Số dòng").to_csv(ROOT / "dataset" / "cleaning_log.csv", encoding="utf-8-sig")

Đã lưu: d:\Bigdata_project\ABDA433708_15_HeThongPhanTichTMDT\dataset\products_clean.csv (6945, 27)
